# Rebuild `smap_by_sample.parquet` safely (shared SMAP cells + low RAM)

This notebook rebuilds `Data/Global/smap/smap_by_sample.parquet` from the six GEE CSV exports.

## Logic from the uploaded SMAP notebooks

The notebooks build `station_smap_grid_mapping.parquet` with `smap_x_idx`, `smap_y_idx`, and `smap_distance_m`. The actual cell identity is the pair **(`smap_x_idx`, `smap_y_idx`)**. If multiple stations have the same two indices, their observations come from the same SMAP grid cell.

This notebook:
1. maps each CSV row's `station_key` to its SMAP grid cell;
2. consolidates rows to one value per **grid cell + date**, and stops if values for the same cell/date conflict;
3. transfers that cell/date value to **all stations** mapped to that grid cell;
4. left-joins values onto `station_date_candidates.parquet`, preserving every candidate row.

## Memory strategy

It does **not** load the full candidate Parquet or all CSV data into pandas. DuckDB performs the joins and writes the result directly to Parquet, with a memory cap and disk spilling. Ensure enough free disk space for the temporary workspace and output.

Run cells from top to bottom. Review shared-cell and data-quality diagnostics before the final write.


In [1]:
from pathlib import Path
from datetime import datetime
import csv
import shutil
import tempfile
import os

try:
    import duckdb
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "duckdb"])
    import duckdb

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Global" or not (PROJECT_DIR / "station_date_candidates.parquet").exists():
    candidates = [
        Path.home() / "Code/UG_SM/Data/Global",
        Path.cwd() / "Data/Global",
    ]
    PROJECT_DIR = next((p for p in candidates if (p / "station_date_candidates.parquet").exists()), PROJECT_DIR)

SMAP_DIR = PROJECT_DIR / "smap"
SAMPLES_PATH = PROJECT_DIR / "station_date_candidates.parquet"
MAPPING_PATH = SMAP_DIR / "station_smap_grid_mapping.parquet"
OUTPUT_PATH = SMAP_DIR / "smap_by_sample.parquet"
OUTPUT_TMP = SMAP_DIR / "smap_by_sample.rebuild_tmp.parquet"

print("Project directory:", PROJECT_DIR.resolve())
print("Sample Parquet exists:", SAMPLES_PATH.exists())
print("Station/grid mapping exists:", MAPPING_PATH.exists())
print("Output target:", OUTPUT_PATH.resolve())

if not SAMPLES_PATH.exists():
    raise FileNotFoundError(f"Not found: {SAMPLES_PATH}")
if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"Not found: {MAPPING_PATH}")


Project directory: /home/mayur/Code/UG_SM/Data/Global
Sample Parquet exists: True
Station/grid mapping exists: True
Output target: /home/mayur/Code/UG_SM/Data/Global/smap/smap_by_sample.parquet


In [2]:
# Discover only CSVs containing the expected GEE export columns.
# This avoids treating unrelated report/error CSVs as inputs.
required_csv_columns = {"date", "sm_surface", "smap_n", "station_key"}
csv_files = []

for path in sorted(SMAP_DIR.glob("*.csv")):
    try:
        with path.open("r", newline="", encoding="utf-8-sig") as f:
            headers = set(next(csv.reader(f), []))
        if required_csv_columns.issubset(headers):
            csv_files.append(path)
    except (OSError, UnicodeDecodeError, StopIteration):
        continue

print(f"Matching GEE CSV exports found: {len(csv_files)}")
for path in csv_files:
    print(f"  {path.name} ({path.stat().st_size:,} bytes)")

if len(csv_files) != 6:
    raise ValueError(
        f"Expected the 6 GEE CSV exports with columns {sorted(required_csv_columns)}, "
        f"but found {len(csv_files)}. Check the directory listing above before continuing."
    )


Matching GEE CSV exports found: 6
  SMAP_L4_DAILY_2015_2020_B01.csv (195,840,307 bytes)
  SMAP_L4_DAILY_2015_2020_B02.csv (200,990,692 bytes)
  SMAP_L4_DAILY_2015_2020_B03.csv (41,550,431 bytes)
  SMAP_L4_DAILY_2020_end_B01.csv (255,156,238 bytes)
  SMAP_L4_DAILY_2020_end_B02.csv (261,880,382 bytes)
  SMAP_L4_DAILY_2020_end_B03.csv (54,139,516 bytes)


In [3]:
# Create an isolated DuckDB work area. It is deleted after a successful run.
# Lower this to '512MB' and/or one thread if your machine has very little RAM.
MEMORY_LIMIT = "768MB"
DUCKDB_THREADS = 2

WORK_DIR = Path(tempfile.mkdtemp(prefix=".smap_rebuild_work_", dir=SMAP_DIR))
WORK_DB = WORK_DIR / "smap_rebuild.duckdb"
DUCKDB_TEMP_DIR = WORK_DIR / "spill"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

con = duckdb.connect(str(WORK_DB))
con.execute(f"SET memory_limit = {sql_literal(MEMORY_LIMIT)}")
con.execute(f"SET temp_directory = {sql_literal(DUCKDB_TEMP_DIR.resolve())}")
con.execute(f"SET threads = {int(DUCKDB_THREADS)}")
con.execute("SET TimeZone = 'UTC'")

print("DuckDB version:", duckdb.__version__)
print("Memory cap:", MEMORY_LIMIT)
print("Threads:", DUCKDB_THREADS)
print("Temporary workspace:", WORK_DIR.resolve())


DuckDB version: 1.5.6
Memory cap: 768MB
Threads: 2
Temporary workspace: /home/mayur/Code/UG_SM/Data/Global/smap/.smap_rebuild_work_gig8756k


In [4]:
# Build a one-row-per-station mapping table.
# The station_key convention follows the uploaded notebooks:
# station_key = network + "::" + station_id.
con.execute(f'''
CREATE OR REPLACE TABLE station_map AS
SELECT
    TRIM(CAST(network AS VARCHAR)) AS network,
    TRIM(CAST(station_id AS VARCHAR)) AS station_id,
    TRIM(CAST(station_key AS VARCHAR)) AS station_key,
    TRY_CAST(smap_x_idx AS INTEGER) AS pixel_x,
    TRY_CAST(smap_y_idx AS INTEGER) AS pixel_y,
    TRY_CAST(smap_distance_m AS DOUBLE) AS station_to_cell_center_m
FROM read_parquet({sql_literal(MAPPING_PATH.resolve())})
''')

mapping_rows = con.execute("SELECT COUNT(*) FROM station_map").fetchone()[0]
mapping_dupe_keys = con.execute('''
    SELECT COUNT(*) FROM (
      SELECT station_key FROM station_map
      GROUP BY station_key HAVING COUNT(*) > 1
    )
''').fetchone()[0]
mapping_dupe_station_ids = con.execute('''
    SELECT COUNT(*) FROM (
      SELECT network, station_id FROM station_map
      GROUP BY network, station_id HAVING COUNT(*) > 1
    )
''').fetchone()[0]
invalid_pixels = con.execute('''
    SELECT COUNT(*) FROM station_map
    WHERE pixel_x IS NULL OR pixel_y IS NULL OR station_key IS NULL OR station_key = ''
''').fetchone()[0]

print(f"Mapping rows: {mapping_rows:,}")
print(f"Duplicated station_key groups: {mapping_dupe_keys:,}")
print(f"Duplicated network/station_id groups: {mapping_dupe_station_ids:,}")
print(f"Stations with missing key/pixel index: {invalid_pixels:,}")

if mapping_dupe_keys or mapping_dupe_station_ids or invalid_pixels:
    raise ValueError("The station/grid mapping is not one-to-one or has missing pixel indices. Inspect the mapping first.")


Mapping rows: 2,210
Duplicated station_key groups: 0
Duplicated network/station_id groups: 0
Stations with missing key/pixel index: 0


In [5]:
# Diagnose co-located stations by exact grid-cell indices.
cell_summary = con.execute('''
WITH cell_counts AS (
    SELECT pixel_x, pixel_y, COUNT(*) AS n_stations
    FROM station_map
    GROUP BY pixel_x, pixel_y
)
SELECT
    (SELECT COUNT(*) FROM station_map) AS stations,
    COUNT(*) AS unique_grid_cells,
    SUM(CASE WHEN n_stations > 1 THEN 1 ELSE 0 END) AS shared_cells,
    SUM(CASE WHEN n_stations > 1 THEN n_stations ELSE 0 END) AS stations_in_shared_cells,
    MAX(n_stations) AS max_stations_per_cell
FROM cell_counts
''').fetchdf()
display(cell_summary)

shared_cells = con.execute('''
SELECT
    pixel_x,
    pixel_y,
    COUNT(*) AS n_stations,
    STRING_AGG(station_key, ' | ' ORDER BY station_key) AS station_keys,
    ROUND(MIN(station_to_cell_center_m), 1) AS min_cell_center_distance_m,
    ROUND(MAX(station_to_cell_center_m), 1) AS max_cell_center_distance_m
FROM station_map
GROUP BY pixel_x, pixel_y
HAVING COUNT(*) > 1
ORDER BY n_stations DESC, pixel_y, pixel_x
LIMIT 100
''').fetchdf()

print("First 100 shared cells (if any):")
display(shared_cells)


,stations,unique_grid_cells,shared_cells,stations_in_shared_cells,max_stations_per_cell
0,2210,1503,178.0,885.0,75


First 100 shared cells (if any):


,pixel_x,pixel_y,n_stations,station_keys,min_cell_center_distance_m,max_cell_center_distance_m
0,2090,206,75,COSMOS::COSMOS__Petzenkirchen | GROW::GROW__0e...,3128.4,5591.1
1,2204,279,39,GROW::GROW__0j1g6k7t | GROW::GROW__0wmxrc70 | ...,4462.7,5053.4
2,2148,208,36,GROW::GROW__1ke3njhh | GROW::GROW__40nmrsbk | ...,290.4,5222.5
3,632,307,30,SOILSCAPE::SOILSCAPE__node1017 | SOILSCAPE::SO...,4671.9,5133.3
4,632,306,26,COSMOS::COSMOS__Tonzi_Ranch | FLUXNET-AMERIFLU...,2700.6,4658.5
...,...,...,...,...,...,...
95,1936,266,2,XMS-CAT::XMS-CAT__Clot_de_les_Peres_I | XMS-CA...,4327.9,4605.5
96,732,269,2,SNOTEL::SNOTEL__Tony_Grove_Lake | SNOTEL::SNOT...,4466.2,5175.9
97,733,269,2,COSMOS::COSMOS__Daniel_Forest | SNOTEL::SNOTEL...,1610.8,1973.0
98,1161,273,2,USCRN::USCRN__Kingston_1_NW | USCRN::USCRN__Ki...,1888.7,2260.9


In [6]:
# Read the six CSV exports into a disk-backed DuckDB table.
# all_varchar prevents auto-inference from altering station IDs; convert needed fields explicitly.
csv_list_sql = "[" + ", ".join(sql_literal(p.resolve()) for p in csv_files) + "]"

con.execute(f'''
CREATE OR REPLACE TABLE smap_raw AS
SELECT
    NULLIF(TRIM(CAST(station_key AS VARCHAR)), '') AS station_key,
    CAST(TRY_CAST("date" AS TIMESTAMPTZ) AS DATE) AS obs_date,
    CASE
        WHEN ISFINITE(TRY_CAST(sm_surface AS DOUBLE))
        THEN TRY_CAST(sm_surface AS DOUBLE)
        ELSE NULL
    END AS sm_surface,
    TRY_CAST(smap_n AS BIGINT) AS smap_n
FROM read_csv_auto(
    {csv_list_sql},
    union_by_name = true,
    all_varchar = true
)
''')

raw_stats = con.execute('''
SELECT
    COUNT(*) AS csv_rows,
    SUM(CASE WHEN station_key IS NULL THEN 1 ELSE 0 END) AS missing_station_key_rows,
    SUM(CASE WHEN obs_date IS NULL THEN 1 ELSE 0 END) AS invalid_date_rows,
    SUM(CASE WHEN sm_surface IS NULL THEN 1 ELSE 0 END) AS missing_or_nonfinite_sm_surface_rows,
    SUM(CASE WHEN smap_n IS NULL THEN 1 ELSE 0 END) AS missing_or_noninteger_smap_n_rows
FROM smap_raw
''').fetchdf()
display(raw_stats)

invalid_rows = con.execute('''
SELECT COUNT(*) FROM smap_raw
WHERE station_key IS NULL OR obs_date IS NULL
''').fetchone()[0]
if invalid_rows:
    examples = con.execute('''
        SELECT * FROM smap_raw
        WHERE station_key IS NULL OR obs_date IS NULL
        LIMIT 20
    ''').fetchdf()
    display(examples)
    raise ValueError(
        f"{invalid_rows:,} CSV rows have an empty station_key or an unparseable date. "
        "Inspect them before merging; nothing has been written."
    )

unmapped_csv_rows = con.execute('''
SELECT COUNT(*)
FROM smap_raw r
LEFT JOIN station_map m USING (station_key)
WHERE m.station_key IS NULL
''').fetchone()[0]

if unmapped_csv_rows:
    unmatched_keys = con.execute('''
        SELECT r.station_key, COUNT(*) AS rows
        FROM smap_raw r
        LEFT JOIN station_map m USING (station_key)
        WHERE m.station_key IS NULL
        GROUP BY r.station_key
        ORDER BY rows DESC
        LIMIT 30
    ''').fetchdf()
    print(f"CSV rows whose station_key is absent from the mapping: {unmapped_csv_rows:,}")
    display(unmatched_keys)
    raise ValueError("Some CSV station keys cannot be mapped to grid cells. No output has been written.")
else:
    print("All CSV station keys map to a known SMAP grid cell.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,csv_rows,missing_station_key_rows,invalid_date_rows,missing_or_nonfinite_sm_surface_rows,missing_or_noninteger_smap_n_rows
0,9286636,0.0,0.0,0.0,0.0


All CSV station keys map to a known SMAP grid cell.


In [7]:
# Build and check one observation per SMAP cell per date.
# Several CSV rows may represent stations that belong to the same pixel.
con.execute('''
CREATE OR REPLACE TABLE smap_cell_qc AS
SELECT
    m.pixel_x,
    m.pixel_y,
    r.obs_date,
    COUNT(*) AS source_rows,
    COUNT(DISTINCT r.sm_surface) AS distinct_sm_surface_values,
    COUNT(DISTINCT r.smap_n) AS distinct_smap_n_values,
    MIN(r.sm_surface) AS min_sm_surface,
    MAX(r.sm_surface) AS max_sm_surface,
    MIN(r.smap_n) AS min_smap_n,
    MAX(r.smap_n) AS max_smap_n
FROM smap_raw r
JOIN station_map m USING (station_key)
GROUP BY m.pixel_x, m.pixel_y, r.obs_date
''')

conflict_count = con.execute('''
SELECT COUNT(*) FROM smap_cell_qc
WHERE distinct_sm_surface_values > 1
   OR distinct_smap_n_values > 1
''').fetchone()[0]

cell_date_count = con.execute("SELECT COUNT(*) FROM smap_cell_qc").fetchone()[0]
print(f"Unique cell/date combinations in the exports: {cell_date_count:,}")
print(f"Conflicting cell/date combinations: {conflict_count:,}")

if conflict_count:
    conflicts = con.execute('''
        SELECT * FROM smap_cell_qc
        WHERE distinct_sm_surface_values > 1
           OR distinct_smap_n_values > 1
        ORDER BY source_rows DESC
        LIMIT 50
    ''').fetchdf()
    display(conflicts)
    raise ValueError(
        "A single SMAP pixel/date has different values in the CSV exports. "
        "The notebook stopped rather than silently averaging conflicting data."
    )

# With conflicts excluded, MAX returns the single non-null value for each cell/date.
con.execute('''
CREATE OR REPLACE TABLE smap_cells AS
SELECT
    pixel_x,
    pixel_y,
    obs_date,
    MAX(max_sm_surface) AS SMAP,
    MAX(max_smap_n) AS SMAP_n_obs
FROM smap_cell_qc
GROUP BY pixel_x, pixel_y, obs_date
''')

print("Deduplicated cell/date records:", f"{con.execute('SELECT COUNT(*) FROM smap_cells').fetchone()[0]:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique cell/date combinations in the exports: 6,314,408
Conflicting cell/date combinations: 336,233


,pixel_x,pixel_y,obs_date,source_rows,distinct_sm_surface_values,distinct_smap_n_values,min_sm_surface,max_sm_surface,min_smap_n,max_smap_n
0,2204,279,2024-04-03,39,2,1,0.338369,0.352452,8,8
1,2204,279,2024-11-21,39,2,1,0.208764,0.215012,8,8
2,2204,279,2026-06-15,39,2,1,0.258431,0.275094,8,8
3,2204,279,2016-06-28,39,2,1,0.249395,0.252988,8,8
4,2204,279,2016-10-04,39,2,1,0.172222,0.175815,8,8
5,2204,279,2016-10-06,39,2,1,0.178984,0.182684,8,8
6,2204,279,2017-12-01,39,2,1,0.337480,0.347567,8,8
7,2204,279,2018-12-17,39,2,1,0.333097,0.339628,8,8
8,2204,279,2019-04-14,39,2,1,0.388016,0.401749,8,8
9,2204,279,2020-05-10,39,2,1,0.311762,0.328196,8,8


ValueError: A single SMAP pixel/date has different values in the CSV exports. The notebook stopped rather than silently averaging conflicting data.

In [ ]:
# Check the candidate schema and ensure all candidate stations have a grid mapping.
sample_columns = [
    row[0] for row in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())})"
    ).fetchall()
]
required_sample_columns = {"sample_id", "station_id", "network", "sample_date"}
missing_sample_columns = required_sample_columns - set(sample_columns)
if missing_sample_columns:
    raise ValueError(f"Candidate Parquet is missing columns: {sorted(missing_sample_columns)}")
if "SMAP" in sample_columns or "SMAP_n_obs" in sample_columns:
    raise ValueError("Candidate Parquet already has SMAP/SMAP_n_obs columns; handle those columns before rebuilding.")

unmapped_sample_rows = con.execute(f'''
SELECT COUNT(*)
FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())}) s
LEFT JOIN station_map m
  ON TRIM(CAST(s.network AS VARCHAR)) = m.network
 AND TRIM(CAST(s.station_id AS VARCHAR)) = m.station_id
WHERE m.station_key IS NULL
''').fetchone()[0]

print(f"Candidate columns: {len(sample_columns)}")
print(f"Candidate sample rows without a station/grid mapping: {unmapped_sample_rows:,}")

if unmapped_sample_rows:
    unmapped_sample_stations = con.execute(f'''
        SELECT s.network, s.station_id, COUNT(*) AS sample_rows
        FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())}) s
        LEFT JOIN station_map m
          ON TRIM(CAST(s.network AS VARCHAR)) = m.network
         AND TRIM(CAST(s.station_id AS VARCHAR)) = m.station_id
        WHERE m.station_key IS NULL
        GROUP BY s.network, s.station_id
        ORDER BY sample_rows DESC
        LIMIT 30
    ''').fetchdf()
    display(unmapped_sample_stations)
    raise ValueError("Some candidate stations do not have a pixel mapping. No output has been written.")

source_rows = con.execute(
    f"SELECT COUNT(*) FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())})"
).fetchone()[0]
source_distinct_ids = con.execute(
    f"SELECT COUNT(DISTINCT sample_id) FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())})"
).fetchone()[0]
print(f"Candidate rows: {source_rows:,}")
print(f"Distinct sample_id values: {source_distinct_ids:,}")
if source_rows != source_distinct_ids:
    raise ValueError("sample_id is not unique in station_date_candidates.parquet; review before merging.")


In [ ]:
# Write the result directly from SQL. No full candidate DataFrame is built in RAM.
# Keep all original candidate columns and add station_key when the source lacks it.
if OUTPUT_TMP.exists():
    OUTPUT_TMP.unlink()

station_key_select = "" if "station_key" in sample_columns else ", m.station_key AS station_key"

copy_sql = f'''
COPY (
    SELECT
        s.*{station_key_select},
        c.SMAP AS SMAP,
        c.SMAP_n_obs AS SMAP_n_obs
    FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())}) s
    JOIN station_map m
      ON TRIM(CAST(s.network AS VARCHAR)) = m.network
     AND TRIM(CAST(s.station_id AS VARCHAR)) = m.station_id
    LEFT JOIN smap_cells c
      ON c.pixel_x = m.pixel_x
     AND c.pixel_y = m.pixel_y
     AND c.obs_date = CAST(TRY_CAST(s.sample_date AS TIMESTAMPTZ) AS DATE)
) TO {sql_literal(OUTPUT_TMP.resolve())}
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
'''
con.execute(copy_sql)
print("Temporary output written:", OUTPUT_TMP.resolve())
print("Temporary output size:", f"{OUTPUT_TMP.stat().st_size:,} bytes")


In [ ]:
# Validate the temporary Parquet before replacing the target.
tmp_sql = sql_literal(OUTPUT_TMP.resolve())
check = con.execute(f'''
SELECT
    COUNT(*) AS output_rows,
    COUNT(DISTINCT sample_id) AS output_distinct_sample_ids,
    SUM(CASE WHEN SMAP IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_smap,
    SUM(CASE WHEN SMAP IS NULL THEN 1 ELSE 0 END) AS rows_without_smap
FROM read_parquet({tmp_sql})
''').fetchdf()
display(check)

output_rows = int(check.loc[0, "output_rows"])
output_distinct_ids = int(check.loc[0, "output_distinct_sample_ids"])

if output_rows != source_rows:
    raise ValueError(f"Output row count {output_rows:,} does not equal source row count {source_rows:,}. Target was not replaced.")
if output_distinct_ids != source_distinct_ids:
    raise ValueError("Distinct sample_id count changed. Target was not replaced.")

print("Validation passed: all candidate rows and sample IDs are preserved.")


In [ ]:
# Safe finalization: preserve the existing (possibly damaged) output, then replace it.
backup_path = None
if OUTPUT_PATH.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}.backup_{stamp}{OUTPUT_PATH.suffix}")
    shutil.copy2(OUTPUT_PATH, backup_path)
    print("Previous output backed up to:", backup_path.resolve())

OUTPUT_TMP.replace(OUTPUT_PATH)
print("Rebuilt output:", OUTPUT_PATH.resolve())
print("Output rows:", f"{output_rows:,}")
print("Output size:", f"{OUTPUT_PATH.stat().st_size:,} bytes")

con.close()
shutil.rmtree(WORK_DIR, ignore_errors=True)
print("Temporary DuckDB workspace cleaned up.")


## Interpreting the results

- **Shared cells** is the count of unique `(smap_x_idx, smap_y_idx)` pairs assigned to more than one station.
- The **cell/date** check ensures stations sharing a pixel are not assigned conflicting values for the same date.
- The output should have the same row count and distinct `sample_id` count as `station_date_candidates.parquet`. Missing SMAP values can remain where a cell/date is absent from the six exports.
- The result contains the original candidate columns, a derived `station_key` if the source did not have one, and `SMAP` / `SMAP_n_obs`.
- If the kernel crashes again, set `MEMORY_LIMIT = "512MB"`, `DUCKDB_THREADS = 1`, and check free disk space. The notebook avoids materializing the multi-million-row candidate table as a pandas DataFrame.
